# Scale one training run across many devices

You will take the language model from [notebook 05](05-train-a-language-model.ipynb) and run it two ways on eight devices: with replicated parameters (data parallelism, `MeshSpec(fsdp=1)`) and with fully sharded parameters (`MeshSpec(fsdp=8)`). You will print where one weight actually lives in each layout, compare step time between the two, and move a checkpoint written under one mesh onto the other.

There is no second code path to learn. `Trainer` builds a mesh from `MeshSpec`, derives a sharding for every leaf of the train state from `Layout`, and hands both to `jax.jit`; XLA inserts the collectives. `fsdp=1` is the degenerate mesh with one device on the `fsdp` axis, which is plain data parallelism.

**Where this runs.** Any machine can simulate 8 devices on the CPU backend with `--xla_force_host_platform_device_count=8`; `SIMULATE_DEVICES` in the parameters cell sets that before JAX loads, and that is the route executed while writing this notebook. A Colab TPU runtime has 8 chips on one host and runs the notebook as written with `SIMULATE_DEVICES = False`; on a single GPU both meshes degenerate to that one device. Simulated CPU devices report process-wide memory, so the memory comparison only means something on real hardware.

In [ ]:
# Install cell: only runs on Colab (import google.colab succeeds there).
# A TPU runtime gets jax[tpu] instead of jax[cuda12]. Locally this cell does nothing.
import os
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    jax_spec = "jax[tpu]" if "COLAB_TPU_ADDR" in os.environ else "jax[cuda12]"
    %pip install -q "dew-ml @ git+https://github.com/AshishKumar4/dew" {jax_spec}

In [ ]:
import os

SIMULATE_DEVICES = True  # fake 8 host devices so the notebook runs anywhere; set False on a TPU runtime
STEPS = 20               # per fit; enough to see the step time, not to train a language model
BATCH_SIZE = 16          # must divide across the devices on the data axis
SEQUENCE_LENGTH = 32
EMB_FEATURES = 64
NUM_LAYERS = 2
NUM_HEADS = 4
DATA_DIR = "data/07-tokens"
RUN_DIR = "runs/07-scaling"

if SIMULATE_DEVICES:
    # XLA reads these once, when it opens a backend, so they have to be set
    # before the first JAX import anywhere in the process. The device count
    # applies to the CPU backend, so the platform is pinned with it.
    os.environ["JAX_PLATFORMS"] = "cpu"
    os.environ["XLA_FLAGS"] = (
        os.environ.get("XLA_FLAGS", "") + " --xla_force_host_platform_device_count=8").strip()

In [ ]:
import jax
print("devices:", jax.devices())
print("backend:", jax.default_backend())

## The mesh

`build_mesh(MeshSpec(...))` returns a six-axis mesh named `data`, `expert`, `fsdp`, `tensor`, `sequence` and `stage`; the factors you do not set are 1, and the data axis takes whatever devices are left. Batches split across the data and fsdp axes at once; only the parameters distinguish the axes. On an 8-device host, `fsdp=1` is a mesh where every device holds the whole model and a different slice of the batch, and `fsdp=8` is one where every device holds an eighth of every large parameter.

In [ ]:
from dew import MeshSpec
from dew.training.distributed import build_mesh

mesh_dp = build_mesh(MeshSpec(fsdp=1))
mesh_fsdp = build_mesh(MeshSpec(fsdp=8))
print("data-parallel mesh:", dict(mesh_dp.shape))
print("fsdp mesh:", dict(mesh_fsdp.shape))

## Where a parameter lives

`Layout` maps the logical axes the modules declare onto the mesh's parameter axes. Below `min_shard` elements a parameter stays replicated, because splitting it costs more in collectives than it saves in memory; the default is 65,536, which would keep every weight of this small model replicated, so the notebook lowers it to show the placement. `Layout.shardings(mesh, tree)` is the rule applied to a whole tree, and the embedding table (256 vocabulary rows by 64 features) shows the two layouts.

In [ ]:
import jax.numpy as jnp
from dew import Layout

layout = Layout(min_shard=1)
weight = jnp.ones((256, EMB_FEATURES))
for name, mesh in (("fsdp=1", mesh_dp), ("fsdp=8", mesh_fsdp)):
    sharding = layout.shardings(mesh, {"params": {"embed_tokens": {"embedding": weight}}})
    spec = sharding["params"]["embed_tokens"]["embedding"].spec
    print(f"{name}: embedding spec {spec}")
    jax.debug.visualize_array_sharding(
        jax.device_put(weight, sharding["params"]["embed_tokens"]["embedding"]))

## The run

The data is the generated corpus of notebook 05, written again here so this notebook stands alone; `TokenWindows` cuts it into windows and shards the records by process. Two trainers are built over the same model and objective, one per mesh, and each runs `STEPS` steps. The trainer prints the mesh it built, and the state is initialised directly into the target layout, so a model too large for one device is never materialised on one device.

In [ ]:
import json
from pathlib import Path

import numpy as np
from dew.data import Loading, TokenWindows
from dew.data.text import ByteTokenizer

data_dir = Path(DATA_DIR)
data_dir.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(0)
subjects = ["the cat", "a dog", "the bird", "my friend", "the child"]
verbs = ["sees", "likes", "finds", "wants", "hears"]
objects = ["the ball", "a tree", "the river", "some food", "the moon"]
text = "".join(f"{rng.choice(subjects)} {rng.choice(verbs)} {rng.choice(objects)}.\n"
               for _ in range(4000))
tokenizer = ByteTokenizer()
ids = np.asarray(tokenizer.encode(text))
val_len = int(round(len(ids) * 0.02))
ids[:val_len].astype(np.uint8).tofile(data_dir / "val.bin")
ids[val_len:].astype(np.uint8).tofile(data_dir / "train.bin")
meta = {"tokenizer": "byte", "vocab_size": 256, "dtype": "uint8",
        "train_tokens": len(ids) - val_len, "val_tokens": val_len, "eos_id": None}
(data_dir / "meta.json").write_text(json.dumps(meta))

data = TokenWindows(path=DATA_DIR, seq_len=SEQUENCE_LENGTH, val_batches=2,
                    loading=Loading(workers=0, threads=1, read_buffer=2)).load(batch=BATCH_SIZE)
print("training windows:", data.records)

In [ ]:
import optax
from dew import Checkpoints, Trainer, models
from dew.objectives.lm import LMObjective

model = models.build("causal_transformer", vocab_size=meta["vocab_size"],
                     emb_features=EMB_FEATURES, num_layers=NUM_LAYERS, num_heads=NUM_HEADS,
                     max_seq_len=SEQUENCE_LENGTH, dtype="float32", attention_impl="xla")
objective = LMObjective(model, SEQUENCE_LENGTH)


def make_trainer(fsdp, name):
    return Trainer(objective, optax.adamw(1e-3), key=jax.random.key(0),
                   mesh=MeshSpec(fsdp=fsdp), layout=Layout(min_shard=1),
                   checkpoints=Checkpoints(f"{RUN_DIR}/{name}"))


variables = jax.eval_shape(objective.init, jax.random.key(0))
print(f"{sum(int(np.prod(x.shape)) for x in jax.tree_util.tree_leaves(variables)) / 1e6:.2f}M parameters")

## The comparison

Both fits run the same steps on the same data, and the mesh is the only difference. After each fit the embedding table's placement is read off the returned state: `sharding.spec` names the mesh axis each dimension is split over, and `addressable_shards` lists the slice each local device holds. On a TPU or a GPU host the sharded run holds an eighth of each large parameter per device; simulated CPU devices show the same layout while the step time carries XLA's simulation of the collectives.

In [ ]:
import time


def run(trainer, steps):
    start = time.perf_counter()
    state = trainer.fit(data, steps=steps, log_every=steps)
    return state, time.perf_counter() - start


def describe(state, name):
    embedding = state.params["params"]["embed_tokens"]["embedding"]
    print(f"{name}: embedding {embedding.shape} spec {embedding.sharding.spec}")
    for shard in embedding.addressable_shards[:2]:
        print(f"  {shard.device}: rows {shard.index[0]} -> local shape {shard.data.shape}")


dp_state, dp_seconds = run(make_trainer(1, "replicated"), STEPS)
describe(dp_state, "replicated (fsdp=1)")
print(f"  {dp_seconds / STEPS * 1000:.0f} ms/step including compilation")

fsdp_state, fsdp_seconds = run(make_trainer(8, "sharded"), STEPS)
describe(fsdp_state, "sharded (fsdp=8)")
print(f"  {fsdp_seconds / STEPS * 1000:.0f} ms/step including compilation")

## Checkpoints cross meshes

The replicated trainer wrote its final state under `runs/07-scaling/replicated`. The trainer below is built with `fsdp=8` over that same directory, so its state lives sharded, and `place()` restores the checkpoint into its own layout: the restore template comes from the freshly initialised sharded state, and the checkpoint's arrays are reassembled onto whatever mesh asks for them. Training a few more steps shows that what arrived is a model that trains, not just a tree of the right shapes.

In [ ]:
crossed = Trainer(objective, optax.adamw(1e-3), key=jax.random.key(0),
                  mesh=MeshSpec(fsdp=8), layout=Layout(min_shard=1),
                  checkpoints=Checkpoints(f"{RUN_DIR}/replicated"))
restored, _, _ = crossed.place()
describe(restored, "restored onto fsdp=8")
same = all(np.array_equal(np.asarray(a), np.asarray(b))
           for a, b in zip(jax.tree_util.tree_leaves(restored.params),
                           jax.tree_util.tree_leaves(dp_state.params)))
print("restored parameters equal the replicated run's:", same)

continued = crossed.fit(data, steps=STEPS + 5, log_every=5)
print("continued to step", int(continued.step), "on the fsdp mesh")

## What changes on a multi-host pod

Everything above was one process. On a TPU pod slice every host runs the same script and sees the whole slice. The new obligations are joining the process pool before anything else touches JAX, and data and checkpoints every host can reach:

```python
from dew.training.runtime import prepare_process

prepare_process(multi_host=True)   # joins the pool from the cluster environment
```

`prepare_process` calls `jax.distributed.initialize()`, which finds the coordinator from the environment a TPU pod provides. The data specs shard records by process, so each host reads its own part of the dataset with no coordination, and a `gs://` checkpoint directory is written from every host.

The `dew-tpu` command drives it from your laptop; every command takes `--dry-run` to print what it would do:

```bash
dew-tpu create my-slice --type v5e-8          # create the slice, wait for READY
dew-tpu setup my-slice --from-source           # install dew and jax[tpu] on every worker
dew-tpu train my-slice --job lm-1 -- \
    recipes/lm/train.py data:token-windows --data.path /home/you/tokens \
    --trainer.mesh.fsdp 8 --trainer.checkpoint-dir gs://your-bucket/runs
```

`train` syncs the working tree to every worker, starts the recipe on all of them detached with `--trainer.multi-host True`, and follows worker 0's log. Nothing in the recipe changes between one host and eight: `--trainer.mesh.fsdp` picks the mesh, and the batch and the shards are per-process pieces of one global run. None of that was executed here; the [TPU guide](../docs/tpu.md) covers the prerequisites and costs.

## Where to go next

On a slice larger than one host the `data` axis grows past 8 with `fsdp` held at 8, or both grow; `build_mesh` divides whatever device count the pool exposes. `Layout.min_shard` decides which parameters are worth splitting; on a decoder with a large vocabulary the embedding table is the first thing worth sharding. `Trainer(accumulation=k)` trades step time for a bigger effective batch when the batch no longer fits in device memory. The [distributed training guide](../docs/concepts/distributed.md) covers the expert, tensor, sequence and stage axes.